## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the Clinical De-identification model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *clinical_deidentification_docwise_wip_en* offer in the [Azure Marketplace](https://marketplace.azure.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-manage-endpoints) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## Clinical Deidentification Spanish

Deidentification is essential for safeguarding patient privacy in clinical data, including texts, PDFs, images, and DICOM files containing Protected Health Information (PHI). PHI encompasses various health-related data, including common identifiers such as name, address, birth date, and Social Security Number.

- **Model**: [es.deid.clinical](https://nlp.johnsnowlabs.com/2023/06/17/clinical_deidentification_es.html)
- **Model Description**: This pipeline is trained with sciwiki_300d embeddings and can be used to deidentify PHI information from medical texts in Spanish. The PHI information will be masked and obfuscated in the resulting text. The pipeline can mask, fake or obfuscate the following entities: AGE, DATE, PROFESSION, E-MAIL, USERNAME, LOCATION, DOCTOR, HOSPITAL, PATIENT, URL, IP, MEDICALRECORD, IDNUM, ORGANIZATION, PHONE, ZIP, ACCOUNT, SSN, PLATE, SEX and IPADDR

In [ ]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**: 
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }

    ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
    {
        "text": "Single text document"
    }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

### Important Parameter

- **masking_policy**: `str`

    Users can select a masking policy to determine how sensitive entities are handled:

    Example: "**Nombre: Danilo. Apellidos: Ramos Portillo. NHC: 4376372. NASS: 35 61051794 56. Domicilio: Calle Fernando Higueras, 27, 7B.**"

    - **masked**: Default policy that masks entities with their type.

      -> 'Nombre: `<PATIENT>`. Apellidos: `<PATIENT>`. NHC: `<SSN>`. NASS: `<ID>`. Domicilio: `<PATIENT>`, `<AGE>`, 7B.'

    - **obfuscated**: Replaces sensitive entities with random values of the same type.

      -> 'Nombre: `Aurora Garrido Paez`.Apellidos: `Antonio González Cuevas`.NHC: `BBBBBBBBQR648597`.NASS: `48.124.111S`. Domicilio: `Aurora Garrido Paez`, `36`, 7B.'

    - **masked_fixed_length_chars**: Masks entities with a fixed length of asterisks (\*).

      -> 'Nombre: `****`. Apellidos: `****`. NHC: `****`. NASS: `****`. Domicilio: `****`, `****`, 7B.'

    - **masked_with_chars**: Masks entities with asterisks (\*).

      -> 'Nombre: [`****`]. Apellidos: [`************`]. NHC: [`*****`]. NASS: [`************`]. Domicilio: [`*********************`], `**`, 7B.'
    
You can specify these parameters in the input as follows:

```json
{
    "text": [
        "Text document 1",
        "Text document 2"
    ],
    "masking_policy": "masked"
}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.


### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [35]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import requests
import json


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record) if content_type == "application/json" else record,
     
    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return response_body
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.


In [ ]:
docs = [
''' Nombre: David. Apellidos: Flores Blanco. NHC: 8972604. Domicilio:  Calle Puerto Rico, 17, 3 C. Loca... 
 Nombre: David. Apellidos: Flores Blanco. NHC: 8972604. Domicilio:  Calle Puerto Rico, 17, 3 C. Localidad/ Provincia: Albacete. CP: 02541. Datos asistenciales. Fecha de nacimiento: 20/10/2003. País: España. Edad: 13 años Sexo: H. Fecha de Ingreso: 04/01/2017. Médico: María Flores Copete NºCol: 02 02 12630.
Informe clínico del paciente: Niño de 13 años, con antecedentes personales de síndrome de Asperger, trastorno de déficit de atención e hiperactividad e hipotiroidismo, en tratamiento con metilfenidato, risperidona y levotiroxina. Fue enviado a la consulta de Dermatología por presentar una lesión en el miembro inferior derecho, de crecimiento progresivo, desde hacía un año. Era asintomática, sin presentar ninguna clínica como dolor o prurito. No había recibido ningún tratamiento, ni refería ningún factor desencadenante. En la exploración se observó una placa, indurada a la palpación, con un área discrómica central y un halo violáceo, de distribución lineal en el tercio medio-distal en la zona externa del muslo derecho. No presentaba otras lesiones cutáneas, ni clínica sistémica acompañante.
Padre con rasgos de Asperger no diagnosticado.
Se solicitó una analítica que incluía hemograma, bioquímica, anticuerpos antinucleares (ANA) y serologías, examen de orina, así como biopsia de la lesión. Los resultados fueron normales, con ANA, anti-Sm, anti-RNP, anti-SSA, anti-SSB, anti-Jo1 y anti-Scl70 negativos. La serología para Borrelia burgdorferi fue negativa y el análisis de orina no mostró anomalías. La histopatología fue compatible con el diagnóstico de morfea.
Ante una morfea lineal en la infancia sin afectación articular ni deformidad estética, se decidió iniciar tratamiento corticoideo tópico con propionato de fluticasona, una aplicación al día durante 2 meses, seguido de calcipotriol tópico, también un aplicación diaria. Tras casi un año de seguimiento, las lesiones se han estabilizado.
Remitido por: Dra. María Flores Copete. Centro de Salud Zona 4. C/ Seminario, 4. C.P. 02006. Albacete. España. Correo electrónico: florescopetemaria@gmail.com''',

'''Nombre: Alexia. Apellidos: Hernandez Peña. NHC: 5347636/301. Domicilio: Av. Sta. María 281, Recoleta... 
Nombre: Alexia. Apellidos: Hernandez Peña. NHC: 5347636/301. Domicilio: Av. Sta. María 281, Recoleta, Región Metropolitana. Localidad/ Provincia: Santiago.
CP: 8420000. Datos asistenciales. Fecha de nacimiento: 21/06/1971. País: Chile. Edad: 44 años Sexo: Mujer. Fecha de Ingreso: 09/08/2015. Médico: Julio Villanueva Maffei.
Motivo de ingreso: Paciente mujer, 44 años, derivado a nuestra unidad por aumento de volumen en la región cigomática y limitación progresiva de la apertura bucal de 10 meses de evolución. La historia médica no era relevante. 
Exploración: El examen físico extraoral muestra asimetría facial por aumento de volumen en la mejilla izquierda, de limites difusos, consistencia ósea, indoloro, con piel de aspecto normal. La apertura bucal era 30 mm. La articulación temporomandibular (ATM) era normal a la palpación, sin ruidos o dolor en apertura. En la ortopantomografía observamos una apófisis coronoides izquierda de mayor tamaño. La hipótesis diagnóstica fue tumor coronoídeo.
Pruebas complementarias y evolución:  Se realizó una incisión por sobre el borde anterior de la rama mandibular, desinsertando todas las inserciones del músculo temporal. Alrededor del aumento de volumen se encontró una pseudocápsula fibrosa, que fue liberada. Se efectuó la coronoidectomía. La apertura bucal se recuperó inmediatamente hasta 43 mm. El postoperatorio se desarrolló sin complicaciones, y la paciente fue dada de alta a las 48 horas.
El estudio histopatológico informó la presencia fibras, hueso de neoformación y tejido cartilaginoso hialino. El diagnóstico fue osteocondroma. Los controles posteriores se desarrollaron sin problemas. El alta definitiva fue dada a los 10 meses, con remodelación casi completa de la deformidad de la mejilla.
Remitido por: Dr. Julio Villanueva Maffei. Santa María 571, Recoleta Chile E-mail: javm@vtr.net'''

]

sample_text = """Nombre: Marina . Apellidos: Casas Gómez. NHC: 8455635. Domicilio: Calle del Dr. Esquerdo 99, Bajo A ... 
Nombre: Marina . Apellidos: Casas Gómez. NHC: 8455635. Domicilio: Calle del Dr. Esquerdo 99, Bajo A . Localidad/ Provincia: Madrid.
CP: 28007. Datos asistenciales. Fecha de nacimiento: 21/07/2007. País: España. Edad: 7 años Sexo: M. Fecha de Ingreso: 03/11/2014. Servicio/ Unidad: Infectología. Episodio: 524652564. Médico: Miguel Peñafiel Freire  NºCol: 28 28 97102. Historia Actual: Niña de siete años nacida en España y sin patología previa que es remitida a la Unidad de Infectología Pediátrica por adenopatías dolorosas submandibulares izquierdas de un mes de evolución, sin respuesta al tratamiento con amoxicilina-ácido clavulánico y antiinflamatorio oral.
La paciente no había presentado fiebre, sudoración nocturna ni pérdida de peso. No estaba vacunada de BCG. Durante un mes al año visita Marruecos, donde tiene contacto con animales (ovejas, corderos) y consume lácteos sin pasteurizar.
Exploración física: En la exploración física presenta un conglomerado submandibular izquierdo de adenopatías, doloroso a la palpación, duro, no rodadero ni adherido, la mayor de las adenopatías de 2 × 2 cm. El resto de la exploración física y anamnesis por aparatos no presentaba hallazgos patológicos.
Resumen de pruebas complementarias: Dada la falta respuesta al tratamiento y teniendo en cuenta los datos epidemiológicos, se realizó la prueba de tuberculina, que fue positiva, con una induración de 10 mm con patrón vesicular a las 72 horas. La radiografía de tórax y la ecografía abdominal fueron normales.
Las serologías de Brucella, virus de Epstein-Barr (VEB) y citomegalovirus (CMV) fueron negativas. La analítica mostró hemograma normal, velocidad de sedimentación globular (VSG) de 47 mm/hora y el interferon-gamma release assays (IGRA) (QuantiFERON(r)) fue positivo, con un valor de 10,15 (rango normal: 0-0,35).
Se realizó una punción aspiración con aguja fina (PAAF) de la adenopatía, que mostraba un material espeso de coloración blanquecina. El cultivo y el estudio citológico revelaron un patrón de linfadenitis granulomatosa necrotizante.
Evolución y comentarios: Dada la escasa afectación de la paciente y la sospecha de adenitis tuberculosa, se pautó un tratamiento domiciliario inicial con isoniacida, rifampicina y pirazinamida, a la espera del cultivo de la muestra obtenida mediante PAAF. El cultivo fue positivo para M. bovis, presentando un antibiograma con resistencia a pirazinamida y sensibilidad a estreptomicina, etambutol, isoniacida y rifampicina. Por este motivo se sustituyó la pirazinamida por etambutol durante dos meses, continuando posteriormente con isoniacida y rifampicina hasta finalizar seis meses. Los cultivos de micobacterias en jugos gástricos fueron negativos. Durante ese tiempo la paciente acudió a revisiones periódicas en el Servicio de Infectología Pediátrica, en las que se constató una buena evolución clínica y la resolución completa del proceso infeccioso.
Remitido por:  Dr. Miguel Peñafiel Freire. Hospital General Universitario Gregorio Marañón. Calle del Dr. Esquerdo, 46, 28007 Madrid. Correo electrónico: mpfreire.89@gmail.com"""

### JSON

#### Example 1: masked (default-policy)

In [ ]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'Record date : <DATE>, <DOCTOR>, M.D. is <PROFESSION>,  Name: <PATIENT>. Age: <AGE>. Phone: (<AGE>) <AGE>5-5632. MR. # <MEDICALRECORD> Date: <DATE> PCP: <DOCTOR>. Record date: <DATE>. <HOSPITAL> <STREET>. This <AGE>-yr-old male, presented with chest heaviness that started during a <PROFESSION>. His past medical history was unremarkable. <PATIENT> denied prior cardiac symptoms and suffered no chest trauma during the game. His father had suffered an acute myocardial infarction at age <AGE>. Elvis <PATIENT> was a nonsmoker, did not drink alcohol, and denied recreational drug use. <PATIENT> swallowed a tablet of aspirin before coming to the emergency room. His blood pressure was 160/90 mm Hg, and his heart rate was 80 bpm. Physical examination revealed no stigmata of Marfan syndrome. The rest of his physical examination was normal. DD: <DATE> . DV: <DATE>, Cocke <LOCATION> <STREET>. PHONE : <PHONE>'}

#### Example 2: obfuscated

In [ ]:
input_json_data = {"text": sample_text, "masking_policy": "obfuscated"}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'Record date : 2013-03-10, Monroe Hoot, M.D. is dentist,  Name: Ballard Kemps. Age: 18. Phone: (8) 7765-5632. MR. # 6827447 Date: 03-10-1993 PCP: Rosalia. Record date: 2013-01-04. Shawnee Mission Prairie Star 1540 Maple Rd. This 18-yr-old male, presented with chest heaviness that started during a psychiatrist. His past medical history was unremarkable. Ballard Kemps denied prior cardiac symptoms and suffered no chest trauma during the game. His father had suffered an acute myocardial infarction at age 32. Elvis Kemps was a nonsmoker, did not drink alcohol, and denied recreational drug use. Ballard Kemps swallowed a tablet of aspirin before coming to the emergency room. His blood pressure was 160/90 mm Hg, and his heart rate was 80 bpm. Physical examination revealed no stigmata of Marfan syndrome. The rest of his physical examination was normal. DD: 03/10/2013 . DV: 03/10/2013, Cocke 1409 9th Street 1540 Maple Rd. PHONE : (+8) 714-310-1155'}

### JSON Lines

In [39]:
def create_jsonl(records, masking_policy=None):
    json_records = []

    if isinstance(records, str):
        records = [records]

    for text in records:
        record = {"text": text}

        if masking_policy is not None:
            record["masking_policy"] = masking_policy
        json_records.append(record)

    json_lines = '\n'.join(json.dumps(record) for record in json_records)
    return json_lines


#### Example 1: masked (default-policy)

In [40]:
input_jsonl_data = create_jsonl(docs, masking_policy="masked")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": "Mr. <PATIENT>, <AGE> years-old , born in <CITY>, was transfered to the The <HOSPITAL>. Phone number: <PHONE>. MSW: <PHONE> . for his colonic polyps.  <PATIENT> wants to know the results from them.  He is not taking hydrochlorothiazide and is curious about his blood pressure. He said he has cut his alcohol back to 6 pack once a week.  <PATIENT> has cut back his cigarettes to one time per week. MR. # <IDNUM> P:   Follow up with Dr. <CITY> <CITY> in 3 months. <DOCTOR> P. <DOCTOR>, M.D. <HOSPITAL> <STREET>. PHONE: <PHONE>"}
{"predictions": "Record date : <DATE> , <DOCTOR> , M.D . , Name : <DOCTOR>. <NAME>: <EMAIL>. MR # <MEDICALRECORD>  Phone : <PHONE>. Date : <DATE> PCP : <DOCTOR> , <AGE> years-old. A long history of insulin dependent diabetes, type 2; coronary artery disease; chronic renal insufficiency; peripheral vascular disease, also secondary to diabetes; who was originally admitted to an outside hospital for what appeared to be acute paraplegia, lower extremities. 

#### Example 2: obfuscated

In [41]:
input_jsonl_data = create_jsonl(docs, masking_policy="obfuscated")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": "Mr. Salena Meals, 39 years-old , born in Olathe, was transfered to the The Northside Medical Center. Phone number: (078) 607-4585. MSW: 855552341091 . for his colonic polyps.  Salena Meals wants to know the results from them.  He is not taking hydrochlorothiazide and is curious about his blood pressure. He said he has cut his alcohol back to 6 pack once a week.  Salena Meals has cut back his cigarettes to one time per week. MR. # 05681172 P:   Follow up with Dr. Avoca Magee in 3 months. Shellie P. Consuela, M.D. Northside Medical Center 1118 S Farmerville St. PHONE: (+8) 785-200-0555"}
{"predictions": "Record date : 2022-03-10 , Rogerio Hoot , M.D . , Name : Pasquale Grit. DARLYNE: felicitas@yahoo.com. MR # 6825447  Phone : (000) 000-8181. Date : 03-10-1993 PCP : Rosalia , 23 years-old. A long history of insulin dependent diabetes, type 2; coronary artery disease; chronic renal insufficiency; peripheral vascular disease, also secondary to diabetes; who was originally a